# 06 · 教程与功能总览

本 notebook 有三重定位:

1. **简明教学** — 按 01→05 号 notebook 的顺序走一遍完整工作流,
   每个环节配一段说明文字;
2. **功能展示** — 用真实算例把前 5 个 notebook 的**全部**功能各
   演示一遍 (每章标题标明对应哪个 notebook);
3. **黄金验证** — 最后一键复现 9 个官方算例并与归档结果比对。

目录:

| 章 | 内容 | 对应 notebook |
|----|------|--------------|
| §1 | 生成初始束团 (输入卡 + 束团预览) | 01_generator |
| §2 | 追踪设置与运行 (输入卡 + 输出清单) | 02_astra |
| §3 | 相空间/统计/切片/BFF/导出 (最全) | 03_postpro |
| §4 | 演化曲线: 九图/光学函数/Sigma/扫描 | 04_lineplot |
| §5 | 场图: 1D/3D/等离子体/阴极轮廓 | 05_fieldplot |
| §6 | 9 算例一键复现 + 黄金比对 | — |

所用数据: §1-§4 主演示用 Manual_Example 真实运行; 孔径/PScan/
Scan 用归档 golden; 少数 ASTRA 不常开的文件类型 (Xemit2/TRemit/
Cemit/Larmor/Error/tcheck/track/Cathode) 用**合成演示数据**并明确
标注, 教学其读法与画法。

## §0 · 准备: 算例目录与运行机制

下面定义了 9 个官方算例的文件清单/运行步骤/黄金比对目标, 以及
stage_files (只备文件) / run_example (备文件+全流程) / compare
(黄金比对) 三个工具。DESY 官方 9 算例: Manual(手册主例)/Aperture
(孔径)/Wake(尾场)/Cavity(腔与TWS)/Curved_Cathode(弯曲阴极)/
90deg_bend(弯转)/Plasma_1(束驱动尾场)/Plasma_2(激光驱动, 需本地
laser.dat)。

In [7]:
%run _bootstrap.py

astra-notebook 后端已加载 (v0.1.0)
项目根目录: /Users/yuxinwu/my_projects/astra_notebook
模拟工作目录: /Users/yuxinwu/my_projects/astra_notebook/data/workspace
ASTRA    : /Users/yuxinwu/programs/ASTRA/astra
Generator: /Users/yuxinwu/programs/ASTRA/generator


In [ ]:
# ===== 共享: 9 算例定义 + 运行/比对机制 =====
import shutil, json
from pathlib import Path
from astra_tools.run import run_program
from astra_tools.io.astra_emit import parse_output_file
from astra_tools.io.field_map import fix_laser_map_header

EXAMPLES_DIR = PROJECT_ROOT / "examples"
GOLDEN_EXPECTED = json.loads((EXAMPLES_DIR / "golden_expected.json").read_text())

EXAMPLES = {
    "Manual_Example": dict(
        copy=["generator.in", "Example.in", "3_cell_L-Band.dat", "Solenoid.dat"],
        patch={"Example.in": (
            "/Users/yuxinwu/astra_notebook/simulation_files/Example.ini",
            "Example.ini")},
        steps=[("generator", "generator.in"), ("astra", "Example.in")],
        golden_xemit=EXAMPLES_DIR / "Manual_Example/Example.Xemit.001"),
    "Aperture": dict(
        copy=["astra.in", "aperture.in", "Geometry.dat", "test.ini"],
        steps=[("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Aperture/golden/astra.Xemit.001"),
    "Wake": dict(
        copy=["Wake.in", "test.ini", "TESLA_MODULE_WAKE_TAYLOR.dat", "test.dat"],
        src_dir="Wake/Wake_Files",
        steps=[("astra", "Wake.in")],
        golden_xemit=EXAMPLES_DIR / "Wake/golden/Wake.Xemit.001"),
    "Cavity_Example": dict(
        copy=["generator.in", "astra.in", "TWS_Sband.dat", "3_cell_L-Band.dat",
              "dcfield.dat", "3D_test.bx", "3D_test.by", "3D_test.bz"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Cavity_Example/golden/astra.Xemit.001"),
    "Curved_Cathode_Example": dict(
        copy=["generator.in", "astra.in", "Contour.dat", "efld.dat"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Curved_Cathode_Example/golden/astra.Xemit.001"),
    "90deg_bend_Example": dict(
        copy=["Section1.in", "Section2.in", "test.ini",
              "3D_Dipole.bx", "3D_Dipole.by", "3D_Dipole.bz"],
        patch={"Section2.in": ("Section1_n.0100.001", "Section1.0100.001")},
        steps=[("astra", "Section1.in"), ("astra", "Section2.in")],
        golden_xemit=EXAMPLES_DIR / "90deg_bend_Example/golden/Section2.Log.001",
        compare_mode="log"),
    "Plasma_Example_1": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt"],
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_1/golden/plasma.Xemit.001"),
    "Plasma_Example_2": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt", "laser.dat"],
        laser_fix=True,
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_2/golden/plasma.Xemit.001"),
}


def stage_files(name):
    """复制官方算例输入到工作目录 (不运行), 返回 (work, spec)。"""
    spec = EXAMPLES[name]
    src = EXAMPLES_DIR / spec.get("src_dir", name)
    work = SIM_DIR / name
    work.mkdir(parents=True, exist_ok=True)
    for f in spec["copy"]:
        s = src / f
        if not s.exists():
            raise FileNotFoundError("%s 缺失: %s" % (f, s))
        shutil.copy2(s, work / f)
    for deck, (old, new) in spec.get("patch", {}).items():
        p = work / deck
        p.write_text(p.read_text().replace(old, new))
    if spec.get("laser_fix"):
        fix_laser_map_header(work / "laser.dat")
        print("  laser.dat 图头计数已转为整数形式")
    return work, spec


def run_example(name):
    """备文件 + 跑完 spec 里全部步骤 (generator/astra)。"""
    work, spec = stage_files(name)
    for kind, deck in spec["steps"]:
        exe = GENERATOR_EXE if kind == "generator" else ASTRA_EXE
        if exe is None:
            raise RuntimeError("未找到 %s 可执行文件" % kind)
        run_program(exe, work, input_file=deck)
    return work


def compare_xemit(name, work):
    """新运行 vs 归档 golden 的末行比对 (rel < 0.5% 判 OK)。"""
    spec = EXAMPLES[name]
    golden = spec["golden_xemit"]
    new_file = work / golden.name
    if not new_file.exists():
        print("  (无 %s 输出, 跳过比对)" % golden.name)
        return
    if spec.get("compare_mode") == "log":
        ok_new = "finished" in new_file.read_text()
        ok_ref = "finished" in golden.read_text()
        print("  末行比对 (new vs golden, log):")
        print("    finished        %-10s %-10s %s"
              % (ok_new, ok_ref, "OK" if ok_new and ok_ref else "MISMATCH"))
        return
    new = parse_output_file(new_file)
    ref = parse_output_file(golden)
    print("  末行比对 (new vs golden):")
    for key in ("norm_emit_x", "sigma_x", "mean_z"):
        a = float(__import__("numpy").asarray(new[key])[-1])
        b = float(__import__("numpy").asarray(ref[key])[-1])
        rel = abs(a - b) / abs(b) * 100
        print("    %-14s %-10.6g %-10.6g rel=%.4f%% %s"
              % (key, a, b, rel, "OK" if rel < 0.5 else "MISMATCH"))


## §1 · 初始束团生成 (对应 01_generator.ipynb)

**01_generator 平时怎么用**: 表单点选 INPUT namelist (Dist_x/Nemit/
sig_z ...), 生成束团并预览。这里用 Manual_Example 的 generator.in
演示同一数据流: 输入卡 → 运行 Generator → 束团预览 (统计表 +
分布图)。

In [ ]:
# ---- 输入卡 generator.in (节选): 束团参数一目了然 ----
work, spec = stage_files("Manual_Example")
print("".join((work / "generator.in").read_text().splitlines(True)[:26]))

# ---- 运行 Generator (只跑 generator, astra 留给 §2) ----
run_program(GENERATOR_EXE, work, input_file="generator.in")
print("Generator 完成, 束团文件: Example.ini")

In [ ]:
# ---- 束团预览: 初始束团的统计表与分布 ----
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html
from astra_tools.plot.distributions import plot_distributions, plot_energy_distribution

ini = read_distribution(work / "Example.ini")   # generator 产物
print_statistics(compute_statistics(ini), title="初始束团 (generator 产物)")
stats_table_html(compute_statistics(ini))
plot_distributions(ini, title_prefix="初始束团")
plot_energy_distribution(ini)

## §2 · 追踪设置与运行 (对应 02_astra.ipynb)

**02_astra 平时怎么用**: 表单生成 astra.in (或直接用现成 .in), 
运行并列出输出。这里用文本模式: 打印输入卡 → 运行 ASTRA → 
输出清单。

In [ ]:
# ---- 输入卡 astra.in (节选): NEWRUN/OUTPUT/CHARGE/SOLENOID ----
print("".join((work / "Example.in").read_text().splitlines(True)[:22]))

# ---- 运行 ASTRA (流式日志) ----
run_program(ASTRA_EXE, work, input_file="Example.in")
print("ASTRA 完成。")

In [ ]:
# ---- 输出清单 (02_astra 的 discover_outputs) ----
import pandas as pd
from astra_tools.run import discover_outputs
outs = discover_outputs(work, "Example", run="001")
rows = []
for key, val in outs.items():
    if isinstance(val, list):
        rows += [(key, str(f.name), f.stat().st_size) for f in val]
    elif val is not None:
        rows += [(key, str(val.name), val.stat().st_size)]
pd.DataFrame(rows, columns=["类型", "文件", "大小(字节)"])

## §3 · 相空间分析与统计 (对应 03_postpro.ipynb)

**03_postpro 平时怎么用**: 选择器挑 z 位置 → 统计表/相空间图/
切片/BFF/导出。本章用 §2 产物在 z = 1.5 m 的相空间文件依次演示
全部功能。

In [ ]:
# ---- 统计表 (与 ASTRA Xemit 交叉验证 <0.02%) ----
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html

dist = read_distribution(work / "Example.0150.001")   # z = 1.5 m 相空间
print_statistics(compute_statistics(dist), title="Manual_Example @ z=1.5 m")
stats_table_html(compute_statistics(dist))

In [ ]:
# ---- 相空间图: 横/纵向 + 归一化视图 ----
from astra_tools.plot.phase_space import plot_transverse_phase_space, plot_phase_space

# 1 GeV 束流的发散角只有 ~1 urad, 原始 x-x' 图必然是一条贴地横线
# (这是物理, 与 Xemit 完全一致); normalize=True 除以各自 sigma 后
# 结构清晰可见
plot_transverse_phase_space(dist, show_ellipse=True)
plot_phase_space(dist, plane="x", show_ellipse=True, normalize=True,
                 title="x-x' normalized")
plot_phase_space(dist, plane="z")

In [ ]:
# ---- 6D 全景 + 横向剖面 + z-plot (含丢失粒子) ----
from astra_tools.plot.overview import plot_overview, plot_transverse_profile
from astra_tools.plot.advanced_plots import plot_z_plot
plot_overview(dist)
plot_transverse_profile(dist)
plot_z_plot(dist)

In [ ]:
# ---- 相空间切割 (postpro 5.6.4): 修改窗口后重跑本单元 ----
from astra_tools.analysis.cuts import cut_distribution
dist_cut, mask = cut_distribution(dist, x_range=(-1e-3, 1e-3))
print("切割后 (x ±1 mm): 保留 %d/%d 粒子" % (dist_cut.n_active, dist.n_active))
stats_table_html(compute_statistics(dist_cut))

In [ ]:
# ---- 纵向切片: 电流/slice 发射度/啁啾 + 失配 + 3D 椭圆 ----
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_dashboard
from astra_tools.plot.advanced_plots import plot_slice_mismatch, plot_slice_ellipses_3d
sa = compute_slice_analysis(dist, n_slices=20)
plot_slice_dashboard(sa)
plot_slice_mismatch(dist, n_slices=20)
plot_slice_ellipses_3d(dist, n_slices=10)

In [ ]:
# ---- 束团形状因子 BFF (相干辐射谱, 直接法 + FFT 快速路径) ----
from astra_tools.analysis.bff import compute_bff
from astra_tools.plot.bff_plots import plot_bff_with_amplitude
bff = compute_bff(dist.filter_active().z, dist.filter_active().charge,
                  kmin=10, kmax=1e5, nk=150, detect_features=True)
plot_bff_with_amplitude(bff)

### 孔径叠加与核心曲线 (03_postpro 的 Aperture/核心功能)

In [ ]:
# ---- 束包络 + 孔径几何叠加 (Aperture 算例, golden 演化数据) ----
from astra_tools.namelist.parse import parse_namelists
from astra_tools.io.astra_emit import read_emit_files
from astra_tools.plot.advanced_plots import aperture_elements, plot_envelope_with_aperture
ap = parse_namelists(PROJECT_ROOT / "examples/Aperture/astra.in")["APERTURE"]
emit_ap = read_emit_files(str(PROJECT_ROOT / "examples/Aperture/golden/astra"))
plot_envelope_with_aperture(emit_ap, aperture_elements(ap))

In [ ]:
# ---- 核心电荷分数曲线 (核心束长/发射度 vs 电荷分数) ----
from astra_tools.plot.advanced_plots import plot_core_fraction_curves
plot_core_fraction_curves(dist)

In [ ]:
# ---- 数据导出: CSV (带单位表头) + npz ----
from astra_tools.export import export_distribution, export_statistics, export_emit
from astra_tools.io.astra_emit import read_emit_files
emit = read_emit_files(str(work / "Example"))
out = SIM_DIR / "export"
print("导出目录:", out)
print("  分布:", export_distribution(dist, out)["csv"].name)
print("  统计:", export_statistics(compute_statistics(dist), out).name)
print("  演化:", ", ".join(p.name for p in export_emit(emit, out).values()))

## §4 · 演化曲线 (对应 04_lineplot.ipynb)

**04_lineplot 平时怎么用**: 读 Xemit/Zemit/ref/Sigma 等演化文件, 
画九图/光学函数/扫描。本章依次: 九图总览 → t 轴变体 → 速度/步长
→ 光学函数 → Sigma 特征发射度 (3.83 之谜已破解) → PScan/Scan 真实
扫描 → 稀有文件类型的读法与画法 (合成教学数据)。

In [ ]:
# ---- 九图总览 (lineplot 菜单 1) ----
from astra_tools.io.astra_emit import read_emit_files, read_ref_file
from astra_tools.plot.emit_plots import plot_lineplot_overview
emit = read_emit_files(str(work / "Example"))
ref = read_ref_file(str(work / "Example"))
plot_lineplot_overview(emit)

In [ ]:
# ---- 时间轴变体 (三视图 vs 时间) + 速度/平均步长 ----
from astra_tools.plot.emit_plots import (plot_envelope_evolution,
    plot_emittance_evolution, plot_velocity_evolution, plot_step_size_evolution)
plot_envelope_evolution(emit, x_axis="t")
plot_emittance_evolution(emit, x_axis="t")
plot_velocity_evolution(ref)
plot_step_size_evolution(ref)

In [ ]:
# ---- 光学函数: beta/alpha、相位推进、相干长度 ----
from astra_tools.plot.advanced_plots import (
    plot_beta_alpha, plot_phase_advance, plot_coherence_length)
plot_beta_alpha(emit)
plot_phase_advance(emit)
plot_coherence_length(emit)

In [ ]:
# ---- Sigma 特征发射度: "3.83 因子"之谜已破解 ----
# Sigma 文件把动量列归一化到 mc、能量列归一化到 mc^2 (1/mc^2=3.83);
# 读者已换算 SI, 导出归一化 eigen-emittance, 与 Xemit 对照 <8%
from astra_tools.io.astra_emit import read_sigma_file
from astra_tools.plot.emit_plots import plot_eigen_emittances
sigma = read_sigma_file(str(work / "Example"))
plot_eigen_emittances(sigma)

### 相位扫描 PScan 与参数扫描 Scan (真实 ASTRA 输出)

In [ ]:
# ---- PScan: 单腔能量增益余弦律 (Cavity_Example golden) ----
from astra_tools.io.astra_misc import read_pscan
from astra_tools.plot.advanced_plots import (
    plot_phase_scan, plot_pscan_dedz, plot_pscan_compression)
pscan = read_pscan(PROJECT_ROOT / "examples/Cavity_Example/golden/astra.PScan.001")
plot_phase_scan(pscan)      # E(phi) = E0 + A cos(phi-phi0), 拟合残差<0.1%A
plot_pscan_dedz(pscan)      # 正比于关联能散
plot_pscan_compression(pscan)

In [ ]:
# ---- Scan: 参数扫描 + lab 标签文件 (Manual_Example golden) ----
from astra_tools.io.astra_misc import read_scan, read_lab_file
from astra_tools.plot.advanced_plots import plot_scan_fom
scan = read_scan(PROJECT_ROOT / "examples/Manual_Example/Example.Scan.001")
lab = read_lab_file(PROJECT_ROOT / "examples/Manual_Example/golden/Example.lab.001")
plot_scan_fom(scan, i=0, lab=lab)   # FOM(1) 水平发射度 vs 螺线管场
plot_scan_fom(scan, i=1, lab=lab)   # FOM(2) 束长
plot_scan_fom(scan, i=2, lab=lab)   # FOM(3) 平均能量

### 稀有文件类型 (合成教学数据: 读法与画法与真实一致)

Xemit2(缩减发射度)/TRemit(trace)/Cemit(核心)/Larmor(拉莫尔角)/
Error(误差扫描)/tcheck(空间电荷缩放)/LandF(损失)/track(探针轨迹)/
Cathode(阴极发射) 只有在 deck 打开对应开关时 ASTRA 才输出; 这里用
合成数据演示, 真实文件把路径换掉即可。

In [ ]:
import numpy as np
from astra_tools.io.astra_emit import parse_output_file
z = np.linspace(0, 1.5, 20)
x2 = dict(z=z, K2z=np.zeros(20), K3z=np.zeros(20),
          eps_red_z=np.linspace(0.9e-6, 1e-6, 20),
          K2E=np.zeros(20), K3E=np.zeros(20),
          eps_red_zE=np.linspace(1.2e-6, 1.1e-6, 20))
tr = dict(z=z, t=np.linspace(0, 5e-9, 20),
          eps_tr_x=np.linspace(1e-6, 1.1e-6, 20),
          eps_tr_y=np.linspace(1e-6, 0.9e-6, 20),
          eps_tr_z=np.linspace(1e-6, 1.2e-6, 20))
lm = dict(z=z, avr=np.linspace(0, 0.5, 20), rms=np.linspace(0.1, 0.3, 20))
tc = dict(z=z, scaling=np.random.default_rng(0).uniform(0.5, 1.5, (20, 5)))
err = dict(run=np.arange(30), z=np.full(30, 1.5),
           FOM=np.random.default_rng(1).normal(1e-6, 1e-8, (30, 10)))
landf = dict(landf_z=z, landf_n_particles=np.full(20, 500.0),
             landf_total_charge=np.full(20, 1e-9), landf_n_lost=np.zeros(20),
             landf_energy_deposited=np.linspace(0, 1e-4, 20),
             landf_energy_exchange=np.linspace(0, 2e-5, 20))
track = dict(seq=np.repeat([1, 2, 3], 20), status=np.ones(60, int),
             z=np.tile(z, 3), x=np.tile(np.linspace(0, 1e-3, 20), 3),
             y=np.tile(np.linspace(0, -0.5e-3, 20), 3),
             Ez=np.linspace(0, 1e6, 60), Er=np.linspace(1e5, 0, 60),
             Ey=np.linspace(0, 5e4, 60))
cathode = dict(t=np.linspace(0, 1e-9, 20), E_acc=np.linspace(5e6, 3e6, 20),
               E_spch=np.linspace(-2e6, 0, 20), q=np.linspace(0, 1e-3, 20))
# Cemit: 13 列, 走真实解析器
crows = np.column_stack([z] + [np.full(20, v) for v in
    (1.0, 0.95, 0.9, 0.85, 1.0, 0.95, 0.9, 0.85, 5.0, 4.8, 4.5, 4.2)])
np.savetxt(SIM_DIR / "demo.Cemit.001", crows)
ce = parse_output_file(SIM_DIR / "demo.Cemit.001")
print("合成教学数据就绪 (Xemit2/TRemit/Cemit/Larmor/Error/tcheck/LandF/track/Cathode)")

In [ ]:
from astra_tools.plot.advanced_plots import (
    plot_reduced_emittance, plot_trace_emittance, plot_core_emittance,
    plot_core_brightness, plot_larmor, plot_error_hist, plot_tcheck_scaling,
    plot_losses, plot_beam_loading, plot_probe_trajectories,
    plot_space_charge_fields, plot_cathode_emission)
plot_reduced_emittance(x2, x2)
plot_trace_emittance(tr)
plot_core_emittance(ce)
plot_core_brightness(ce, landf)
plot_larmor(lm)
plot_error_hist(err, i=0)
plot_tcheck_scaling(tc)
plot_losses(landf)
plot_beam_loading(landf)
plot_probe_trajectories(track)
plot_space_charge_fields(track)
plot_cathode_emission(cathode)

## §5 · 场图 (对应 05_fieldplot.ipynb)

**05_fieldplot 平时怎么用**: 读腔场/螺线管/3D 场图/等离子体剖面并
画场图。本章展示 1D 场、3D 场图截面与轴上剖面、等离子体密度剖面、
弯曲阴极轮廓。

In [ ]:
# ---- 1D 腔场 (离轴展开) 与螺线管场 ----
from astra_tools.io.field_map import read_cavity_field, read_solenoid_field
from astra_tools.plot.field_plots import plot_cavity_field, plot_solenoid_field
cav = read_cavity_field(PROJECT_ROOT / "examples/Manual_Example/3_cell_L-Band.dat")
sol = read_solenoid_field(PROJECT_ROOT / "examples/Manual_Example/Solenoid.dat").scaled(0.35)
plot_cavity_field(cav, omega=2 * np.pi * 1.3e9)
plot_solenoid_field(sol)

In [ ]:
# ---- 3D 场图截面 + 轴上剖面 (Cavity_Example 的 3D_test) ----
from astra_tools.plot.advanced_plots import plot_3d_map_slices, plot_laser_on_axis
plot_3d_map_slices(PROJECT_ROOT / "examples/Cavity_Example/3D_test.ex",
                   axis="z", n_slices=3, unit="V/m")
plot_laser_on_axis(PROJECT_ROOT / "examples/Cavity_Example/3D_test.ex", unit="V/m")

In [ ]:
# ---- 等离子体密度剖面 + 弯曲阴极轮廓 ----
from astra_tools.plot.advanced_plots import (
    plot_plasma_profile, plot_curved_cathode_contour)
plot_plasma_profile(PROJECT_ROOT / "examples/Plasma_Example_1/PLASMA_flattop.txt",
                    peak_density_cm3=1e17)
plot_curved_cathode_contour(PROJECT_ROOT / "examples/Curved_Cathode_Example/Contour.dat")

## §6 · 一键复现 9 个官方算例 + 黄金比对

约 1 分钟。任何一行 FAILED 或 MISMATCH 都说明本机环境或后端有
问题 (完整回归测试见 test/ 与 docs/dev_manual/test_plan.md)。

In [ ]:
for name in EXAMPLES:
    print("=" * 60)
    print(name)
    try:
        w = run_example(name)
        compare_xemit(name, w)
    except Exception as e:
        print("  FAILED:", e)
    print()

## 下一步

* 想深入某一环节: 打开对应编号的 notebook (01-05), 参数表单交互式
  探索;
* 想导出数据自行绘图: 任何 notebook 都可用 astra_tools.export;
* 完整文档: docs/user_guide/ (用户手册), docs/dev_manual/
  (开发手册与测试方案), docs/physics_notes/ (物理约定备忘录)。